# Free benchmark run on Google Colab

Runs `scripts/run_real_models.sh` with local models through Ollama on Colab's free GPU: no API key, $0.

1. **Runtime > Change runtime type > T4 GPU**, then run the cells in order.
2. Results and the response cache are written to Google Drive, so when Colab disconnects (free sessions are time-limited) you re-run all cells and the run **resumes** where it stopped; finished model calls are served from the cache.
3. Start with the quick check (one benchmark, one repeat) to see how fast your session is, then run the full plan.

In [ ]:
# Install and start Ollama
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
import time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull llama3.1
!ollama pull qwen2.5:7b

In [ ]:
# Keep results and the cache on Google Drive so a disconnect loses nothing
import os

from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/agent-eval"
os.makedirs(BASE, exist_ok=True)
os.environ["AGENT_EVAL_CACHE_DIR"] = f"{BASE}/cache"
os.environ["AGENT_EVAL_DATA_DIR"] = f"{BASE}/data"
os.environ["OUT"] = f"{BASE}/results"
os.environ["YES"] = "1"  # no interactive prompt in a notebook; the default models cost $0

In [ ]:
# Get the harness
!test -d Agent-Eval-Harness || git clone https://github.com/LasyaRamachandruni/Agent-Eval-Harness.git
%cd Agent-Eval-Harness
!git pull --ff-only
!pip install -q -e .

In [ ]:
# Quick check first: built-in tasks, one repeat, react only (shows how long a run takes here)
!BENCHMARKS=builtin STRATEGIES=react REPEATS=1 scripts/run_real_models.sh

In [ ]:
# Full free plan: llama3.1 + qwen2.5:7b x builtin, HotpotQA-200, HumanEval x 4 strategies.
# REPEATS=3 keeps it shorter on Colab; use 5 for the full pass^5 numbers. Re-run this cell to resume.
!REPEATS=3 scripts/run_real_models.sh

In [ ]:
# Where the reports are
!ls $OUT/*/report.html $OUT/*/leaderboard.md
!cat $OUT/*/leaderboard.md